# Squat-Only YOLO Pose Extraction (Google Colab)

This notebook keeps the local pipeline unchanged and provides a separate Colab workflow for running squat-only pose extraction with a GPU.

## 1. Select a GPU Runtime

In Colab:
- `Runtime` -> `Change runtime type`
- `Hardware accelerator` -> `GPU`

In [ ]:
import os

# Choose ONE of the options below.
# Option A: clone from GitHub
REPO_URL = "<YOUR_REPO_URL>"
WORKDIR = "/content/personal-git"

# Option B: use Google Drive
# WORKDIR = "/content/drive/MyDrive/<YOUR_PATH>/personal-git"

## 2. Get the Repo into Colab

In [ ]:
# Run this cell if you want to clone from GitHub.
!git clone $REPO_URL /content/personal-git

In [ ]:
# Run this cell instead if the repo is already in Google Drive.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd $WORKDIR
!pwd
!ls CV_Image_pose_detection/artifacts/3_Modeling

## 3. Install Pose Dependencies

In [ ]:
!python3 -m pip install -r CV_Image_pose_detection/requirements-pose.txt

## 4. Confirm GPU

In [ ]:
import torch

print("cuda_available =", torch.cuda.is_available())
print("device_count =", torch.cuda.device_count())
print("device_name =", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 5. Build a Squat-Only Index

In [ ]:
!python3 CV_Image_pose_detection/artifacts/3_Modeling/build_pose_feature_index.py \
  --exercise squat \
  --output-csv CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index_squat.csv

In [ ]:
!head -n 5 CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index_squat.csv

## 6. Smoke Test on 5 Squat Videos

In [ ]:
!python3 CV_Image_pose_detection/artifacts/3_Modeling/pose_feature_extraction.py \
  --index-csv CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index_squat.csv \
  --video-dir CV_Image_pose_detection/Data/LLSP/video \
  --device cuda:0 \
  --max-videos 5 \
  --overwrite

## 7. Run the Full Squat Set

Remove `--overwrite` if you want to resume and skip already-finished files.

In [ ]:
!python3 CV_Image_pose_detection/artifacts/3_Modeling/pose_feature_extraction.py \
  --index-csv CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index_squat.csv \
  --video-dir CV_Image_pose_detection/Data/LLSP/video \
  --device cuda:0

## 8. Inspect Outputs

In [ ]:
import json
from pathlib import Path

summary_path = Path("CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_summary.json")
print(json.loads(summary_path.read_text()))

In [ ]:
import pandas as pd

report = pd.read_csv("CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_report.csv")
report.head()